In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# wczytanie danych
spam_dataset = pd.read_csv('spam.csv', encoding="ISO-8859-1", usecols=[0, 1], names=['Spam', 'Text'],skiprows=1)

spam_dataset['Spam'] = spam_dataset['Spam'].replace(['ham', 'spam'], [0, 1])
spam_dataset

,Spam,Text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [3]:
X = spam_dataset['Text'].astype(str)
y = spam_dataset['Spam'].astype('int64').to_numpy().ravel()

print(np.unique(y))
print(y.dtype)

[0 1]
int64


In [4]:
# Podzial na zbior treningowy i testowy ze stratyfikacja
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y)

In [5]:
# Wektoryzacja TF-IDF
tfidf = TfidfVectorizer(
    ngram_range=(2, 2),
    max_df=0.5,
    min_df=0.001,
    use_idf=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [6]:
# Pierwszy Random Forest - bez selekcji cech

clf = RandomForestClassifier(max_depth=2, random_state=0)
clf.fit(X_train_tfidf, y_train)

print("Score pierwszego modelu:")
print(clf.score(X_test_tfidf, y_test))

Score pierwszego modelu:
0.8663677130044843


In [7]:
# Feature importance - wybiera, ktore slowa sa najwazniejsze po >0.001

feature_importances = clf.feature_importances_
feature_names = np.array(tfidf.get_feature_names_out())

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances}).sort_values(by='importance', ascending=False)

print(feature_importance_df.head(20))

                feature  importance
1269         to contact    0.057384
273               co uk    0.040971
1266           to claim    0.039466
1603        your mobile    0.030173
1376        urgent your    0.024477
530          holiday or    0.023502
296         contact you    0.021670
978             out who    0.021570
373            find out    0.020961
658           just call    0.020686
1008             po box    0.020386
957             or call    0.019664
26          2nd attempt    0.017999
998         please call    0.017759
54    account statement    0.015642
1126            stop to    0.015056
665            just txt    0.014937
268          claim call    0.014045
1253           to 86688    0.013519
501            have won    0.013194


In [8]:
# Selekcja cech (importance > 0.001)
selected_features = feature_importance_df[
    feature_importance_df['importance'] > 0.001]['feature'].values

print("Liczba wybranych cech:")
print(len(selected_features))

Liczba wybranych cech:
107


In [9]:
# indeksy wybranych cech
selected_features_idx = np.where(np.isin(feature_names, selected_features))[0]

X_train_selected = X_train_tfidf[:, selected_features_idx]
X_test_selected = X_test_tfidf[:, selected_features_idx]

In [10]:
# Drugi Random Forest + Gridsearch
# Gridsearch szuka najleszych parametrow
clf_v2 = RandomForestClassifier(random_state=0)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']}

grid = GridSearchCV(
    estimator=clf_v2,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1)

grid.fit(X_train_selected, y_train)

print("Najlepsze parametry:")
print(grid.best_params_)

print("Najlepszy wynik CV:")
print(grid.best_score_)

Najlepsze parametry:
{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Najlepszy wynik CV:
0.9557993990729419


In [11]:
# sprawdzenie na zbiorze testowym
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_selected)

print("Accuracy na zbiorze testowym:")
print(accuracy_score(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Accuracy na zbiorze testowym:
0.9488789237668162
Classification report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       966
           1       0.97      0.64      0.77       149

    accuracy                           0.95      1115
   macro avg       0.96      0.82      0.87      1115
weighted avg       0.95      0.95      0.94      1115



In [12]:
# Countvectorizer

count = CountVectorizer(
    min_df=0.001,
    max_df=0.5,
    ngram_range=(2, 2))

X_train_count = count.fit_transform(X_train)
X_test_count = count.transform(X_test)

clf_count = RandomForestClassifier(
    max_depth=2,
    random_state=0)

clf_count.fit(X_train_count, y_train)

print("Score dla CountVectorizer:")
print(clf_count.score(X_test_count, y_test))

print("Vocabulary CountVectorizer:")
print(count.vocabulary_)

Score dla CountVectorizer:
0.8663677130044843
Vocabulary CountVectorizer:
{'but have': 190, 'have to': 499, 'to pay': 1297, 'got it': 458, 'wan me': 1399, 'now or': 895, 'co uk': 273, 'or call': 957, 'you must': 1556, 'lt gt': 739, 'am in': 73, 'had your': 479, 'your mobile': 1603, 'update to': 1360, 'to the': 1313, 'the latest': 1185, 'latest camera': 687, 'for free': 380, 'text yes': 1149, 'watching tv': 1413, 'am not': 75, 'at home': 138, 'more than': 811, 'find out': 373, 'from your': 412, 'see you': 1055, 'where are': 1460, 'are you': 126, 'you now': 1560, 'you will': 1587, 'will take': 1480, 'care of': 256, 'of me': 911, 'we have': 1420, 'have any': 489, 'congrats mobile': 291, 'mobile 3g': 802, '3g videophones': 31, 'videophones yours': 1387, 'yours call': 1612, 'now videochat': 900, 'videochat wid': 1386, 'ur mates': 1368, 'mates play': 756, 'play java': 997, 'java games': 654, 'games dload': 413, 'music noline': 818, 'noline rentl': 875, 'bx420 ip4': 205, 'ip4 5we': 596, 'you 

# porownanie modeli

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Model TF-IDF przed selekcj cech
y_pred_tfidf = clf.predict(X_test_tfidf)

wynik_tfidf = accuracy_score(y_test, y_pred_tfidf)
precision_tfidf = precision_score(y_test, y_pred_tfidf)
recall_tfidf = recall_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf)


# model TF-IDF po selekcji cech i GridSearch
y_pred_tfidf_grid = best_model.predict(X_test_selected)

wynik_tfidf_grid = accuracy_score(y_test, y_pred_tfidf_grid)
precision_tfidf_grid = precision_score(y_test, y_pred_tfidf_grid)
recall_tfidf_grid = recall_score(y_test, y_pred_tfidf_grid)
f1_tfidf_grid = f1_score(y_test, y_pred_tfidf_grid)


#Model CountVectorizer
y_pred_count = clf_count.predict(X_test_count)

wynik_count = accuracy_score(y_test, y_pred_count)
precision_count = precision_score(y_test, y_pred_count)
recall_count = recall_score(y_test, y_pred_count)
f1_count = f1_score(y_test, y_pred_count)


# Tabela do porownania
porownanie = pd.DataFrame({
    'Model': [
        'Random Forest + TF-IDF',
        'Random Forest + TF-IDF + selekcja cech + GridSearch',
        'Random Forest + CountVectorizer'],
    
    'Accuracy': [
        wynik_tfidf,
        wynik_tfidf_grid,
        wynik_count],
    
    'Precision': [
        precision_tfidf,
        precision_tfidf_grid,
        precision_count],
    
    'Recall': [
        recall_tfidf,
        recall_tfidf_grid,
        recall_count],
    
    'F1-score': [
        f1_tfidf,
        f1_tfidf_grid,
        f1_count]})

porownanie.sort_values(by='F1-score', ascending=False)

C:\Users\atrel\anaconda3\envs\tf\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\atrel\anaconda3\envs\tf\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Model,Accuracy,Precision,Recall,F1-score
1,Random Forest + TF-IDF + selekcja cech + GridS...,0.948879,0.969388,0.637584,0.769231
0,Random Forest + TF-IDF,0.866368,0.000000,0.000000,0.000000
2,Random Forest + CountVectorizer,0.866368,0.000000,0.000000,0.000000
